# Users DB → partitioned search index (cloud-only, no localhost)

Builds the searchable parts from `users_data.zip` and uploads each part to its
own account. See `PARTS_README.md` for the full design.

1. Run setup.
2. **Run the speed test** — if the source delivers < ~1 MB/s the full build is
   not practical; get a faster mirror first.
3. Fill the config cell and run the build.

In [ ]:
!apt-get -qq install -y gh >/dev/null 2>&1
!rm -rf users-shards 2>/dev/null; git clone -q https://github.com/namanyash2009-crypto/users-shards.git
%cd users-shards
print('ready')

## 1. Speed test (do this first)

Paste a fresh direct link below and run. `speed_Bps` is bytes/second from the
cloud. You want **≥ ~1,000,000** (1 MB/s). Below ~150 KB/s the 117 GB build
would take more than a week — find a faster mirror.

In [ ]:
LINK = 'https://flowvideoplayer.com/api/v2/download/direct?url=https%3A%2F%2F1024terabox.com%2Fs%2F1ahJz-qdH7h_9One0lXxDoA&idx=0'
!curl -sL --max-time 40 -r 0-50000000 -o /dev/null -w "http=%{http_code} bytes=%{size_download} time=%{time_total} speed_Bps=%{speed_download}\n" "$LINK"

In [ ]:
import os, json

SOURCE = LINK            # reuse the link from the speed-test cell
PARTS = 10               # = number of accounts you will use
EST_TOTAL = 1710000000

# One upload command per part (index 0 -> part-000).
# GitHub example (create the releases first):
#   UPLOAD_CMDS = [f'gh release upload v1 {{dir}}/* --repo ACCT{i}/REPO --clobber' for i in range(PARTS)]
#   os.system('echo TOKEN | gh auth login --with-token')
# Internet Archive example (needs `ia` configured per account):
#   UPLOAD_CMDS = [f'ia upload users-db-part-{i:03d} {{dir}}/*' for i in range(PARTS)]
UPLOAD_CMDS = []         # leave empty to just write ./out/part-XXX locally

os.environ.update(
    SOURCE=SOURCE, PARTS=str(PARTS), EST_TOTAL=str(EST_TOTAL),
    MOB_BUCKETS='64', MAIL_BUCKETS='64', NAME_BUCKETS='64', BLOCK='2048',
    OUT='./out', GZIP_STAGE='1', DELETE_AFTER_UPLOAD='1' if UPLOAD_CMDS else '0',
)
if UPLOAD_CMDS:
    os.environ['UPLOAD_CMDS'] = json.dumps(UPLOAD_CMDS)
print('parts:', PARTS, '| upload cmds:', len(UPLOAD_CMDS))

In [ ]:
!python parts_builder.py

When it finishes, each `out/part-XXX` has been uploaded (and deleted locally if
`DELETE_AFTER_UPLOAD=1`). Collect the base URLs, e.g.:

* GitHub: `https://github.com/ACCT/repo/releases/download/v1`
* Internet Archive: `https://archive.org/download/ITEM`

Then set them in `parts-edge.js` `DEFAULT_PART_BASES` **in part order** and deploy.